# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 4b — Fidelity under node removal and the data-randomisation control

Two measurements require a definition that does not depend on the architecture or on the
evaluation set used for toxicophore agreement.

**Fidelity.** Masking an atom by zeroing its feature vector does not remove it: the input
projection carries a bias, so a zeroed atom enters message passing as a constant vector
rather than as an absence. Fidelity is therefore recomputed by deleting the selected atoms
and their incident bonds from the graph. Fidelity is also restricted to molecules the model
classifies correctly, since the drop in predicted probability is only interpretable when
the model was confident to begin with.

**Data randomisation.** A model trained on permuted labels is informative only if it has
fitted those labels. Early stopping against a validation set that retains its true labels
selects the epoch at which the model best captures the real relationship, which defeats the
control. Both label sets are therefore permuted and training runs for a fixed number of
epochs, with the training AUC reported as evidence of how far the permuted labels were
fitted.

This notebook rewrites `table11_explainers.csv` and `table14_label_permuted.csv`.

*Approximate runtime: 20-30 minutes.*

## 1. Environment

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q shap
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 51.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR = f"{WORK}/data"
OUTDIR  = f"{WORK}/results"
MODELS  = f"{WORK}/models"
EXPL    = f"{WORK}/explain"
for d in (WORK, DATADIR, OUTDIR, MODELS, EXPL):
    os.makedirs(d, exist_ok=True)

import sys
assert os.path.exists("/content/ames_lib.py") or True
print("Working directory:", WORK)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline


The shared module is written again so that this notebook runs on a fresh runtime.

In [3]:
%%writefile /content/ames_lib.py
"""Shared definitions for the Ames mutagenicity pipeline."""

import random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi


def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]


def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr


def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)


def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]


def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f


def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]


def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        ei += [[i, j], [j, i]]
        ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)


def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]


class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = (conv(x, edge_index, edge_attr) if self.use_edge_attr
                 else conv(x, edge_index))
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)


def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads,
               use_edge_attr=use_edge_attr).to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)


def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True):
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)
    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5
        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auc


def train_gnn_fixed(kind, params, seed, tr_set, pos_weight, n_epochs=250,
                    use_edge_attr=True):
    """Train for a fixed number of epochs with no early stopping.

    Used for the data-randomisation control, where the objective is to fit the
    permuted labels rather than to generalise.
    """
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    loader = DataLoader(tr_set, batch_size=params["batch_size"],
                        shuffle=True, drop_last=True)
    for _ in range(n_epochs):
        model.train()
        for d in loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
    return model


class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = (
            x, edge_index, edge_attr, batch)


def remove_atoms(graph, remove_idx):
    """Delete atoms and their incident bonds, returning a renumbered subgraph."""
    n = graph.x.size(0)
    keep = torch.ones(n, dtype=torch.bool)
    if len(remove_idx):
        keep[torch.as_tensor(list(remove_idx), dtype=torch.long)] = False
    if int(keep.sum()) == 0:
        return None
    remap = torch.full((n,), -1, dtype=torch.long)
    remap[keep] = torch.arange(int(keep.sum()))
    ei = graph.edge_index
    if ei.numel():
        keep_e = keep[ei[0]] & keep[ei[1]]
        new_ei = remap[ei[:, keep_e]]
        new_ea = graph.edge_attr[keep_e]
    else:
        new_ei = torch.zeros((2, 0), dtype=torch.long)
        new_ea = torch.zeros((0, BOND_DIM), dtype=torch.float)
    return Data(x=graph.x[keep], edge_index=new_ei, edge_attr=new_ea)

Writing /content/ames_lib.py


In [4]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames_lib import *
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

SEED, PRIMARY_SPLIT_SEED = 42, 101
EXPLAIN_SEED, K_FRAC = 42, 0.30
MAX_EPOCHS, PATIENCE = 250, 30
N_IG_STEPS = 32

HP_GNN     = json.load(open(f"{OUTDIR}/hpo_gnn.json"))
HP_CLASSIC = json.load(open(f"{OUTDIR}/hpo_classic.json"))
print("Device:", DEVICE)

Device: cuda


## 2. Data, representative models and the evaluation set

In [5]:
from xgboost import XGBClassifier

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}")
                for f in ["train.csv", "valid.csv", "test.csv"]],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values

train_set, valid_set, test_set = (build_dataset(df_train), build_dataset(df_valid),
                                  build_dataset(df_test))
valid_loader = DataLoader(valid_set, batch_size=256)
test_loader  = DataLoader(test_set,  batch_size=256)
pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float, device=DEVICE)

hp = HP_GNN["gin"]
gin_rep = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"],
                    hp.get("residual", True))
gin_rep.load_state_dict(torch.load(f"{MODELS}/gin_seed{EXPLAIN_SEED}.pt",
                                   map_location=DEVICE)["state_dict"])
gin_rep.eval()
_, p_gin_valid, _ = evaluate(gin_rep, valid_loader)
_, p_gin_test,  _ = evaluate(gin_rep, test_loader)

xgb_rep = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"],
                        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                        eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                        random_state=EXPLAIN_SEED, n_jobs=-1)
xgb_rep.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
N_TREES = int(xgb_rep.best_iteration) + 1

grid = np.linspace(0.05, 0.95, 91)
THR_GIN = grid[int(np.argmax([balanced_accuracy_score(y_valid, (p_gin_valid >= t).astype(int))
                              for t in grid]))]
print(f"GIN test ROC-AUC {roc_auc_score(y_test, p_gin_test):.4f}, threshold {THR_GIN:.2f}")
print(f"XGBoost test ROC-AUC {roc_auc_score(y_test, xgb_rep.predict_proba(X_test)[:,1]):.4f}, "
      f"{N_TREES} trees")

GIN test ROC-AUC 0.7997, threshold 0.63
XGBoost test ROC-AUC 0.7833, 18 trees


In [6]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}

def toxicophore_atoms(mol):
    atoms = set()
    for patt in TOX_PATTERNS.values():
        for match in mol.GetSubstructMatches(patt):
            atoms.update(match)
    return atoms

EVAL = []
for pos, (smi, lab) in enumerate(zip(df_test["canonical"], y_test)):
    if lab != 1:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0:
        continue
    tox = toxicophore_atoms(mol)
    if not tox or len(tox) == mol.GetNumAtoms():
        continue
    EVAL.append({"row": pos, "smiles": smi, "mol": mol, "tox": tox,
                 "n_atoms": mol.GetNumAtoms(), "chance": len(tox) / mol.GetNumAtoms()})

EVAL_GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
CHANCE, N_EVAL = np.array([e["chance"] for e in EVAL]), len(EVAL)

correct_gin = np.array([p_gin_test[e["row"]] >= THR_GIN for e in EVAL])
print(f"Evaluation set: {N_EVAL} molecules; "
      f"{int(correct_gin.sum())} classified correctly by the representative GIN")

Evaluation set: 237 molecules; 187 classified correctly by the representative GIN


## 3. Fidelity under node removal

Atoms selected for removal are deleted from the graph together with their incident bonds,
so that the molecule seen by the model is genuinely smaller rather than carrying
placeholder nodes.

Fidelity+ is the drop in predicted probability when the most-attributed atoms are removed;
a positive value means those atoms supported the prediction. Fidelity- is the drop when
only those atoms are retained; a value near zero means they were sufficient. The
characterisation score combines the two as a harmonic mean, and the proportion of molecules
with a positive fidelity+ is reported alongside it, since a mean can hide disagreement in
sign across molecules.

In [7]:
def batched_gradient_scores(model, graphs, method="ig", n_steps=N_IG_STEPS,
                            batch_size=64, signed=False):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE)
        x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for alpha in torch.linspace(1.0 / n_steps, 1.0, n_steps):
                xa = (x0 * alpha).detach().requires_grad_(True)
                logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(logit.sum(), xa)[0]
            attr = x0 * (total / n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            grad = torch.autograd.grad(logit.sum(), xa)[0]
            attr = grad if method == "saliency" else x0 * grad
        scores = (attr.sum(dim=1) if signed else attr.abs().sum(dim=1))
        scores = scores.detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for g in range(batch.num_graphs):
            out.append(scores[b == g])
    return out

@torch.no_grad()
def predict_graph(model, graph):
    if graph is None or graph.x.size(0) == 0:
        return None
    b = Batch.from_data_list([graph]).to(DEVICE)
    return float(torch.sigmoid(model(b)).item())

def fidelity_by_removal(model, graphs, scores_list, restrict, k_frac=K_FRAC):
    plus, minus = [], []
    for g, sc, ok in zip(graphs, scores_list, restrict):
        if not ok or sc is None or len(sc) != g.x.size(0):
            continue
        n = g.x.size(0)
        k = max(1, int(round(k_frac * n)))
        if k >= n:
            continue
        top = np.argsort(-np.asarray(sc, dtype=float))[:k]
        rest = np.setdiff1d(np.arange(n), top)
        p_full = predict_graph(model, g)
        p_del  = predict_graph(model, remove_atoms(g, top))    # top-k removed
        p_keep = predict_graph(model, remove_atoms(g, rest))   # only top-k retained
        if None in (p_full, p_del, p_keep):
            continue
        plus.append(p_full - p_del)
        minus.append(abs(p_full - p_keep))
    plus, minus = np.array(plus), np.array(minus)
    fp, fm = float(plus.mean()), float(minus.mean())
    a, b = max(fp, 0.0), max(1.0 - fm, 0.0)
    char = 2 * a * b / (a + b) if (a + b) > 0 else 0.0
    return {"n_fidelity": len(plus), "fidelity_plus": fp, "fidelity_minus": fm,
            "characterisation": char, "pct_fidplus_positive": 100 * (plus > 0).mean()}

gnnex_sc = list(np.load(f"{EXPL}/gnnexplainer_scores.npy", allow_pickle=True))
methods = {"GIN (Integrated Gradients)": batched_gradient_scores(gin_rep, EVAL_GRAPHS, "ig"),
           "GIN (Gradient x Input)":     batched_gradient_scores(gin_rep, EVAL_GRAPHS, "gradinput"),
           "GIN (Saliency)":             batched_gradient_scores(gin_rep, EVAL_GRAPHS, "saliency"),
           "GIN (GNNExplainer)":         gnnex_sc}

rows = []
for label, sc in methods.items():
    r = fidelity_by_removal(gin_rep, EVAL_GRAPHS, sc, correct_gin)
    r["explainer"] = label
    rows.append(r)
fid_df = pd.DataFrame(rows)[["explainer", "n_fidelity", "fidelity_plus",
                             "fidelity_minus", "characterisation",
                             "pct_fidplus_positive"]]
fid_df.to_csv(f"{OUTDIR}/table11b_fidelity_removal.csv", index=False)
print(fid_df.round(4).to_string(index=False))

                 explainer  n_fidelity  fidelity_plus  fidelity_minus  characterisation  pct_fidplus_positive
GIN (Integrated Gradients)         187         0.6395          0.4652            0.5825               95.1872
    GIN (Gradient x Input)         187         0.6455          0.4695            0.5824               97.8610
            GIN (Saliency)         187         0.6467          0.3862            0.6298               96.2567
        GIN (GNNExplainer)         187         0.2403          0.7731            0.2334               90.9091


In [8]:
# Merge the corrected fidelity into the explainer table.
expl = pd.read_csv(f"{OUTDIR}/table11_explainers.csv")
for c in ["fidelity_plus", "fidelity_minus", "characterisation",
          "n_fidelity", "pct_fidplus_positive"]:
    if c in expl.columns:
        expl = expl.drop(columns=[c])
expl = expl.merge(fid_df, on="explainer", how="left")
expl.to_csv(f"{OUTDIR}/table11_explainers.csv", index=False)
print(expl.round(4).to_string(index=False))

                   explainer   n   lift  pct_of_ceiling  attribution_auroc  n_auroc  n_fidelity  fidelity_plus  fidelity_minus  characterisation  pct_fidplus_positive
  XGBoost (TreeSHAP, signed) 237 1.5422         59.2899             0.7979      237         NaN            NaN             NaN               NaN                   NaN
XGBoost (TreeSHAP, absolute) 237 1.4416         48.2902             0.7397      237         NaN            NaN             NaN               NaN                   NaN
  GIN (Integrated Gradients) 237 1.2265         24.7664             0.6150      237       187.0         0.6395          0.4652            0.5825               95.1872
      GIN (Gradient x Input) 237 1.2164         23.6629             0.6051      237       187.0         0.6455          0.4695            0.5824               97.8610
              GIN (Saliency) 237 1.1784         19.5073             0.6132      237       187.0         0.6467          0.3862            0.6298               96.256

## 4. Data-randomisation control

Training and validation labels are both permuted, and each model is fitted for a fixed
budget with no early stopping. The training AUC on the permuted labels records how far each
model fitted the randomised target; the test AUC on the true labels confirms that nothing
generalisable was learned.

In [9]:
import shap
from rdkit.Chem import rdFingerprintGenerator as rfg

def fp_with_bitinfo(mol):
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    fp = morgan_gen.GetFingerprint(mol, additionalOutput=ao)
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr, ao.GetBitInfoMap()

def bit_to_atoms(mol, atom_idx, radius):
    if radius == 0:
        return {atom_idx}
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    atoms = {atom_idx}
    for bidx in env:
        bond = mol.GetBondWithIdx(bidx)
        atoms.add(bond.GetBeginAtomIdx()); atoms.add(bond.GetEndAtomIdx())
    return atoms

BITINFO = []
for e in EVAL:
    try:
        _, bi = fp_with_bitinfo(e["mol"])
    except Exception:
        bi = {}
    BITINFO.append(bi)

def shap_to_atom_scores(shap_values, signed=True):
    sv = np.asarray(shap_values)
    per_molecule = sv.ndim == 2
    out = []
    for i, (e, bi) in enumerate(zip(EVAL, BITINFO)):
        vec = sv[i] if per_molecule else sv
        scores = np.zeros(e["n_atoms"], dtype=float)
        for bit, envs in bi.items():
            v = float(vec[bit])
            if v == 0.0 or len(envs) == 0:
                continue
            for atom_idx, radius in envs:
                atoms = bit_to_atoms(e["mol"], atom_idx, radius)
                share = v / (len(envs) * len(atoms))
                for a in atoms:
                    scores[a] += share
        out.append(scores if signed else np.abs(scores))
    return out

def tree_shap_matrix(model, X):
    values = shap.TreeExplainer(model).shap_values(X)
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    return values

def overlap_from_scores(scores_list, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed)
    out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores_list)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift_from_overlap(ov):
    m = ~np.isnan(ov)
    return float(ov[m].mean() / CHANCE[m].mean()) if m.sum() else np.nan

X_eval = np.array([fp_array(e["smiles"]) for e in EVAL])
rng = np.random.default_rng(SEED)
rows = []

# Graph arm: both label sets permuted, fixed training budget, no early stopping.
perm_tr = rng.permutation(len(train_set))
train_perm = []
for g, j in zip(train_set, perm_tr):
    h = g.clone(); h.y = train_set[j].y.clone(); train_perm.append(h)

m_perm = train_gnn_fixed("gin", HP_GNN["gin"], SEED, train_perm, pos_weight,
                         n_epochs=MAX_EPOCHS)
m_perm.eval()
y_tr_perm = np.array([float(g.y.item()) for g in train_perm])
_, p_tr, _ = evaluate(m_perm, DataLoader(train_perm, batch_size=256))
_, p_te, _ = evaluate(m_perm, test_loader)
ov = overlap_from_scores(batched_gradient_scores(m_perm, EVAL_GRAPHS, "ig"), tie_seed=SEED)
rows.append({"arm": "graph", "model": "GIN (permuted labels)",
             "train_auc_permuted": roc_auc_score(y_tr_perm, p_tr),
             "test_auc_true": roc_auc_score(y_test, p_te),
             "pred_sd": float(np.std(p_te)),
             "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())})
del m_perm; torch.cuda.empty_cache()

# Tree arm: both label sets permuted, fixed number of trees, no early stopping.
y_train_perm = rng.permutation(y_train)
xgb_perm = XGBClassifier(n_estimators=N_TREES, **HP_CLASSIC["XGBoost"],
                         eval_metric="auc", tree_method="hist",
                         random_state=SEED, n_jobs=-1)
xgb_perm.fit(X_train, y_train_perm, verbose=False)
sv = tree_shap_matrix(xgb_perm, X_eval)
ov = overlap_from_scores(shap_to_atom_scores(sv, signed=True), tie_seed=SEED)
rows.append({"arm": "tree", "model": "XGBoost (permuted labels)",
             "train_auc_permuted": roc_auc_score(y_train_perm,
                                                 xgb_perm.predict_proba(X_train)[:, 1]),
             "test_auc_true": roc_auc_score(y_test, xgb_perm.predict_proba(X_test)[:, 1]),
             "pred_sd": float(np.std(xgb_perm.predict_proba(X_test)[:, 1])),
             "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())})

drt_df = pd.DataFrame(rows)
drt_df.to_csv(f"{OUTDIR}/table14_label_permuted.csv", index=False)
print(drt_df.round(4).to_string(index=False))

  arm                     model  train_auc_permuted  test_auc_true  pred_sd   lift   n
graph     GIN (permuted labels)              0.5459         0.5285   0.0289 1.0525 237
 tree XGBoost (permuted labels)              0.6722         0.5040   0.0740 0.8150 237


## 5. Summary

In [10]:
null = pd.read_csv(f"{OUTDIR}/table12_untrained_null.csv")
gin_null = null[null.arch == "gin"].iloc[0]
print("Toxicophore lift")
for _, r in expl.iterrows():
    print(f"  {r.explainer:34s} {r.lift:.4f}")
print(f"  {'untrained GIN, mean of 500':34s} {gin_null['mean']:.4f} "
      f"[{gin_null.p2_5:.4f}, {gin_null.p97_5:.4f}]")
for _, r in drt_df.iterrows():
    print(f"  {r.model:34s} {r.lift:.4f}  "
          f"(permuted-label train AUC {r.train_auc_permuted:.3f}, "
          f"true-label test AUC {r.test_auc_true:.3f})")
print()
print("Fidelity under node removal, correctly classified molecules only")
print(fid_df.round(4).to_string(index=False))

Toxicophore lift
  XGBoost (TreeSHAP, signed)         1.5422
  XGBoost (TreeSHAP, absolute)       1.4416
  GIN (Integrated Gradients)         1.2265
  GIN (Gradient x Input)             1.2164
  GIN (Saliency)                     1.1784
  GIN (GNNExplainer)                 0.8978
  untrained GIN, mean of 500         1.1156 [0.9894, 1.2262]
  GIN (permuted labels)              1.0525  (permuted-label train AUC 0.546, true-label test AUC 0.529)
  XGBoost (permuted labels)          0.8150  (permuted-label train AUC 0.672, true-label test AUC 0.504)

Fidelity under node removal, correctly classified molecules only
                 explainer  n_fidelity  fidelity_plus  fidelity_minus  characterisation  pct_fidplus_positive
GIN (Integrated Gradients)         187         0.6395          0.4652            0.5825               95.1872
    GIN (Gradient x Input)         187         0.6455          0.4695            0.5824               97.8610
            GIN (Saliency)         187         0.646

In [11]:
!cd /content/drive/MyDrive/ames_pipeline && zip -qr /content/part4b_results.zip results
from google.colab import files
files.download("/content/part4b_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>